# Problem 5. Reading Two Lorentzian Peaks

**Day 2 · Problem 5 · 50 points · open-ended.**
**Compute:** `optional-colab-l4`. The CPU path is required and is what the grader runs; the `Accelerator extension` section at the end is optional.

**Allowed tools.** Python's standard library, NumPy, and PyTorch only, on a CPU. No web access, downloads, external datasets, model hubs, pretrained weights, or checkpoints you did not create in the same run.

## The signal model

A **Lorentzian peak** with amplitude $a>0$, centre $c$, and half-width $w>0$ is the function
$$L(x;a,c,w)=\frac{a}{1+\big(\frac{x-c}{w}\big)^2}.$$
It equals $a$ at $x=c$ and $a/2$ at $x=c\pm w$, and its tails decay like $1/x^2$, much more slowly than a Gaussian bump.
Each signal is a sum of exactly $K=2$ peaks,
$$f(x)=L(x;a_1,c_1,w_1)+L(x;a_2,c_2,w_2),$$
with $a_k\in[0.5,2]$, $c_k\in[0.1,0.9]$, $w_k\in[0.02,0.08]$, and centres separated by $|c_1-c_2|\ge1.5\,(w_1+w_2)$.
It is observed at the $40$ midpoints $x_j=(j+0.5)/40$, $j=0,\dots,39$ (`r2_001_data.LORENTZ_X`), with independent additive noise $N(0,0.02^2)$, rounded to four decimals.
`r2_001_data.lorentz_values(x, params)` evaluates the noise-free sum for parameters `(N, 6)`; it is differentiable.

## Task

From the $40$ samples of a signal, recover its six parameters.
Predictions are floating `(N, 6)` tensors $(a_1,c_1,w_1,a_2,c_2,w_2)$, peaks in either order.

**Metric (lower is better).** For one signal, the error is the better of the two peak orders of the mean absolute difference over the six parameters,
$$E=\min_{\pi\in\{(1,2),(2,1)\}}\frac16\sum_{k=1}^{2}\Big(|\hat a_{\pi(k)}-a_k|+|\hat c_{\pi(k)}-c_k|+|\hat w_{\pi(k)}-w_k|\Big),$$
averaged over the signals (`r2_001_data.pair_parameter_error_per_example`).

## Data

* `f, theta = r2_001_data.train_rows("lorentz")`: $800$ signals, `f` float32 `(800, 40)`, `theta` float32 `(800, 6)` $=(a_1,c_1,w_1,a_2,c_2,w_2)$ with the peaks ordered by centre.
* `r2_001_data.val_rows("lorentz")`: $200$ signals, same layout, for selection only.
* Locked test: $200$ signals, reachable only through `final_test_score`.
* You may also draw fresh simulated signals with `r2_001_data.simulate("lorentz", n, seed)` for any `seed != 20261101`.

## Committed baseline

`r2_001_data.baseline_score("lorentz")` fits every signal by least squares (Adam, learning rate $0.02$, $400$ steps, on amplitudes, centres, and log-widths) from the same fixed start: centres $0.35$ and $0.65$, amplitudes $1$, both widths equal to the median training width (`r2_001_data.lorentz_ls_fit` with `lorentz_baseline_init`; both are importable, and `lorentz_ls_fit` accepts an `optimizer_wrapper` argument for the budget).
It scores **B = 0.01711** at the frozen seed.

## Fixed elements (all are graded)

* **Interface and seam.** Your approach exposes `fit(train_x, train_y)` and `predict(x)`; `predict` returns the `(N, 6)` parameters and is the `predict_fn` you pass to `final_test_score`. `fit` records `r2_001_data.split_of("lorentz", row)` for every row of `train_x` into a set `seam`; after fitting, assert that no recorded value is `"val"` or `"test"`. If your method needs no training, `fit` still records the seam and may simply store statistics of the training rows.
* **Budget.** Create `budget = r2_001_data.StepBudget(2000)` before any fitting.
  **Every** optimizer you use, including any per-example refinement inside `predict`, is wrapped with `budget.wrap(...)`; at most 2,000 optimizer steps in total.
  An optimizer over a whole batch of examples costs one step per update, not one per example.
  Only non-iterative closed-form computation is free; any iterative solver must run as a wrapped optimizer so that every iteration counts.
  `budget.elapsed()`, measured after the test call, must be `< 60` seconds on the grading CPU.
* **Protocol.** Fit only on the rows this statement names, make every choice on `r2_001_data.val_rows("lorentz")`, and reach the locked test rows only through `r2_001_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="lorentz")`, called exactly once, after all fitting and selection.
  Assert `r2_001_data.test_call_count("lorentz") == 0` immediately before that call and `== 1` after it, and print the returned `TestScore`.
  `r2_001_data.baseline_score("lorentz")` may be called (it is not counted) to print the baseline under the same protocol.
* **Seam values allowed:** `{"train"}` (plus `None` for signals you simulated yourself).
* **Run clean.** The notebook runs top to bottom in a fresh kernel without errors; delete dead experiments.

## Grading

**Test score (30 points, 60%).** The grader re-executes this notebook top to bottom on a CPU under `SEED = 20261101` and reads the point estimate (`.score`) of your single `final_test_score` call.
The committed baseline scores **B = 0.01711** and the reference solution scores **R = 0.005272** at this seed; let `g = |B - R| = 0.01184`.

* **30 points** (100% of the score portion) if your score is `<= 0.008231` (that is, `<= R + 0.25 g`);
* **18 points** (60%) if it is `< 0.01711` (that is, `< B`);
* **7.5 points** (25%) if it is `< 0.02007` (that is, `< B + 0.25 g`);
* **0 points** otherwise.

A protocol violation (a second `final_test_score` call, a seam value outside the allowed set, the step budget or time limit exceeded, a banned tool, or any use of validation or test rows other than their stated roles) scores **0** for the test-score portion.

**Writeup (20 points, 40%).** Four parts, 5 points each, graded against the rubric descriptors in `rubric.md`:

* **Approach:** the model or method, its inputs and features, and why it suits this problem.
* **Alternatives considered:** at least two, each with the validation evidence that ruled it out.
* **Evaluation:** the validation protocol, the single final test score with its 95% bootstrap interval and `n`, and the comparison with the baseline under the same protocol.
* **Limitations:** at least one failure mode and how you would test for it.

Every number in the writeup must be traceable to an output of this notebook.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()
_spec = importlib.util.spec_from_file_location("r2_001_data", DATA_DIR / "r2_001_data.py")
r2_001_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(r2_001_data)
assert r2_001_data.SEED == SEED

f_train, theta_train = r2_001_data.train_rows("lorentz")
f_val, theta_val = r2_001_data.val_rows("lorentz")
print(f_train.shape, theta_train.shape, f_val.shape)
print("baseline recipe:", r2_001_data.BASELINE_RECIPES["lorentz"])


## Your work

Baseline under the protocol, your validated method, seam, budget, the single test call, and answer-check assertions.

In [ ]:
# Your approach: fit, predict, validation evidence, the single test call, and assertions.


## Writeup

**Approach.** *Replace.*

**Alternatives considered.** *Replace.*

**Evaluation.** *Replace.*

**Limitations.** *Replace.*

## Accelerator extension

*Optional; Colab L4 only; never executed by the grader and never required for credit. The graded score is always the CPU run above.*

On a Colab L4, train an amortized regressor (raw samples in, six parameters out, a permutation-invariant loss) on many simulated signals, then keep a short per-signal least-squares refinement in `float32`:

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "simulated_signals": 500000, "hidden": 512,
             "steps": 30000, "batch_size": 4096, "lr": 1e-3, "refine_steps": 100, "refine_dtype": torch.float32}
```

**What to observe.** Whether the raw regressor beats the CPU pipeline once it has hundreds of thousands of signals, which parameter (amplitude, centre, or width) dominates the remaining error, and how close peaks behave.